# 📊 Notebook 5 – Evaluation (Validation + Test)

**Depends on:** `01_setup.ipynb` + `02_preprocess.ipynb` + `04_pose_train.ipynb`

**What this notebook does:**

| Step | Description | Output |
|------|-------------|--------|
| 5.1 | Load config + model | eval-ready EnhancedRCVPose |
| 5.2 | Define metric functions | translation RMSE, rotation error, ADD |
| 5.3 | Evaluate on **validation** split | per-class + overall table |
| 5.4 | Evaluate on **test** split | final held-out metrics |
| 5.5 | Plot metric distributions | histograms per class |

**Metrics explained:**
| Metric | Formula | Unit |
|--------|---------|------|
| Translation RMSE | √ mean(‖t_pred − t_gt‖²) | metres |
| Rotation Error | 2·arccos(\|q̂_pred·q̂_gt\|) | degrees |
| Points MSE | mean((R_pred − R_gt)²) | m² |
| ADD | mean‖(R_p·m + t_p) − (R_g·m + t_g)‖ | metres |
| ADD Success % | frames with ADD < threshold | % |

> ⚠️ Run the **test** split evaluation only once — after you are done tuning.

## Cell 5.1 – Load config, imports, and best model

We load the best checkpoint path from `config.json` (set by `04_pose_train.ipynb`). The model weights are loaded with `strict=False` to handle the `torch.compile()` prefix (`_orig_mod.`) that appears in compiled model state dicts.

**H100 optimisations applied here:**
- `allow_tf32` on both matmul and cuDNN → ~3× faster matrix ops
- `cudnn.benchmark = True` → finds fastest conv algorithm
- `AMP_DTYPE = bfloat16` on H100 (native hardware support, no overflow risk)
- `_WORKERS = 8` on H100 nodes (more CPU cores available)

In [ ]:
import os, json, sys
import numpy as np
import pandas as pd
import torch
import open3d as o3d
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
from scipy.spatial.transform import Rotation as SciR
from tqdm import tqdm

with open('/content/config.json') as f:
    CONFIG = json.load(f)

DATA_DIR        = CONFIG['DATA_DIR']
ALL_CLASSES     = CONFIG['ALL_CLASSES']
ADD_THRESHOLDS  = CONFIG['ADD_THRESHOLDS']
REPO_DIR        = CONFIG['REPO_DIR']
MODEL_PATH      = CONFIG['BEST_POSE_MODEL']

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.model   import EnhancedRCVPose
from src.dataset import PoseDataset, safe_collate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── H100 / Ampere optimisations ───────────────────────────────────────────
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    # TF32: ~3x faster matmul on Ampere+ with minimal precision loss
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.set_float32_matmul_precision('high')
    # cuDNN benchmark: finds fastest conv kernel for fixed input sizes
    torch.backends.cudnn.benchmark = True
    # BF16 on H100 (native, avoids FP16 overflow), FP16 on older GPUs
    IS_H100   = 'H100' in gpu_name or 'H800' in gpu_name
    AMP_DTYPE = torch.bfloat16 if IS_H100 else torch.float16
    _WORKERS  = 8 if IS_H100 else 4
    print(f'   GPU       : {gpu_name}')
    print(f'   AMP dtype : {AMP_DTYPE}')
    print(f'   TF32      : enabled')
    print(f'   Workers   : {_WORKERS}')
else:
    IS_H100   = False
    AMP_DTYPE = torch.float16
    _WORKERS  = 4

print(f'📂 Loading model: {MODEL_PATH}')
eval_model = EnhancedRCVPose().to(DEVICE)
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
eval_model.load_state_dict(state, strict=False)
eval_model.eval()
print(f'✅ Model loaded (epoch {ckpt.get("epoch", "?")}, val_loss={ckpt.get("val_loss", "?"):.5f})')

## Cell 5.2 – Metric functions

**`translation_rmse`:** Root-Mean-Squared Error of the 3D translation vector. `pred[:3]` and `gt[:3]` are `[tx, ty, tz]` in metres.

**`rotation_error_deg`:** Geodesic angle between two unit quaternions. Uses `|dot|` to handle the quaternion double-cover (q and -q represent the same rotation). Formula: `2 * arccos(|q_pred · q_gt|)` → degrees.

**`add_metric`:** Average Distance of Model points (ADD). Applies both predicted and GT poses to all mesh points, then computes mean point-to-point Euclidean distance. A frame is a success if `ADD < per-class threshold`.

In [ ]:
def translation_rmse(pred: np.ndarray, gt: np.ndarray) -> float:
    """RMSE of 3D translation (metres)."""
    return float(np.sqrt(np.mean((pred[:3] - gt[:3]) ** 2)))


def rotation_error_deg(pred: np.ndarray, gt: np.ndarray) -> float:
    """Geodesic angle (degrees) between predicted and GT quaternion."""
    q_p = pred[3:] / (np.linalg.norm(pred[3:]) + 1e-9)
    q_g = gt[3:]   / (np.linalg.norm(gt[3:])   + 1e-9)
    dot = float(np.clip(np.abs(np.dot(q_p, q_g)), 0.0, 1.0 - 1e-7))
    return float(np.degrees(2.0 * np.arccos(dot)))


def add_metric(pred: np.ndarray, gt: np.ndarray,
               mesh_pts: np.ndarray) -> float:
    """
    ADD metric: mean distance between correspondingly transformed mesh points.
    pred / gt : 7-D [tx,ty,tz, qx,qy,qz,qw]  (metres)
    mesh_pts  : (N, 3)  3D model points in object frame (metres)
    """
    R_p = SciR.from_quat(pred[3:]).as_matrix()
    R_g = SciR.from_quat(gt[3:]).as_matrix()
    pts_p = mesh_pts @ R_p.T + pred[:3]
    pts_g = mesh_pts @ R_g.T + gt[:3]
    return float(np.mean(np.linalg.norm(pts_p - pts_g, axis=1)))


def load_mesh_points(obj_dir: str) -> np.ndarray:
    """Load mesh.ply and return (N,3) points in metres. Returns None if not found."""
    path = os.path.join(obj_dir, 'mesh.ply')
    if not os.path.isfile(path):
        return None
    pts = np.asarray(o3d.io.read_point_cloud(path).points, dtype=np.float32)
    if pts.max() > 1.0:
        pts /= 1000.0    # mm -> m
    return pts


print('✅ Metric functions defined.')

## Cell 5.3 – Evaluate on VALIDATION split

For each class:
1. Build a `PoseDataset(split='val', augment=False)` — **no augmentation**!
2. For each batch: run model inference under `autocast(dtype=AMP_DTYPE)`
3. Compute per-sample: `trans_rmse`, `rot_error`, `pts_mse`, `ADD`
4. Accumulate results and print a formatted table

ADD success counts only frames where `ADD < per-class threshold`.
Mesh points are loaded from `data/XX/mesh.ply` (set up in `02_preprocess.ipynb`).

In [ ]:
def run_evaluation(split: str) -> pd.DataFrame:
    """Run full evaluation on the given split. Returns a results DataFrame."""
    rows = []

    for cls in ALL_CLASSES:
        obj_dir = os.path.join(DATA_DIR, cls)
        try:
            ds = PoseDataset(obj_dir, split=split, num_radius_pts=9, augment=False)
            dl = DataLoader(
                ds, batch_size=1, shuffle=False,
                num_workers=_WORKERS, collate_fn=safe_collate,
                pin_memory=True, persistent_workers=True,
            )
        except FileNotFoundError:
            continue

        mesh_pts = load_mesh_points(obj_dir)
        thresh   = ADD_THRESHOLDS.get(cls, 0.02)

        t_errs, r_errs, p_errs, a_errs = [], [], [], []

        with torch.no_grad():
            for batch in tqdm(dl, desc=f'{split.capitalize()} cls {cls}', leave=False):
                if batch is None:
                    continue
                rgb      = batch['rgb'].to(DEVICE, non_blocking=True)
                depth    = batch['depth'].to(DEVICE, non_blocking=True)
                pose_gt  = batch['pose'].cpu().numpy()[0]
                rmaps_gt = batch['radius_maps'].cpu().numpy()[0]

                with autocast(dtype=AMP_DTYPE):
                    pose_p, rmaps_p = eval_model(rgb, depth)

                pose_p  = pose_p.float().cpu().numpy()[0]
                rmaps_p = rmaps_p.float().cpu().numpy()[0]

                t_errs.append(translation_rmse(pose_p, pose_gt))
                r_errs.append(rotation_error_deg(pose_p, pose_gt))
                p_errs.append(float(np.mean((rmaps_p - rmaps_gt) ** 2)))
                if mesh_pts is not None:
                    a_errs.append(add_metric(pose_p, pose_gt, mesh_pts))

        n_suc = sum(1 for e in a_errs if e < thresh)

        rows.append({
            'Class'         : cls,
            'N'             : len(t_errs),
            'Trans RMSE (m)': round(float(np.mean(t_errs)), 4),
            'Rot Error (deg)': round(float(np.mean(r_errs)), 2),
            'Points MSE'    : round(float(np.mean(p_errs)), 6),
            'ADD (m)'       : round(float(np.mean(a_errs)), 4) if a_errs else None,
            'ADD Success %' : round(100 * n_suc / len(a_errs), 1) if a_errs else None,
        })

    return pd.DataFrame(rows)


print('Running validation evaluation...')
df_val = run_evaluation('val')

print('\n VALIDATION Results')
print('-' * 75)
print(df_val.to_string(index=False))
print('-' * 75)
print(f'  Mean Trans RMSE : {df_val["Trans RMSE (m)"].mean():.4f} m')
print(f'  Mean Rot Error  : {df_val["Rot Error (deg)"].mean():.2f} deg')
if df_val['ADD Success %'].notna().any():
    print(f'  Mean ADD Success: {df_val["ADD Success %"].mean():.1f}%')

## Cell 5.4 – Evaluate on TEST split

> ⚠️ **Run only once!** The test split is the 10% of data held out during preprocessing — never seen during training OR validation.

Running this multiple times and using the results to guide further training constitutes **data leakage** — the test numbers would no longer be trustworthy.

We use the same `run_evaluation()` helper from Cell 5.3, just with `split='test'`.

In [ ]:
print('Running TEST evaluation (held-out set)...')
df_test = run_evaluation('test')

print('\n TEST Results  (final, held-out set)')
print('-' * 75)
print(df_test.to_string(index=False))
print('-' * 75)
print(f'  Mean Trans RMSE : {df_test["Trans RMSE (m)"].mean():.4f} m')
print(f'  Mean Rot Error  : {df_test["Rot Error (deg)"].mean():.2f} deg')
if df_test['ADD Success %'].notna().any():
    print(f'  Mean ADD Success: {df_test["ADD Success %"].mean():.1f}%')

## Cell 5.5 – Plot metric distributions as bar charts

We plot four metrics side-by-side, one bar per class:
1. Translation RMSE (m)
2. Rotation Error (°)
3. Points MSE
4. ADD Success % (higher is better)

Reference lines:
- **Trans RMSE:** red dashed at `0.02 m` (2 cm) — common threshold for tabletop manipulation
- **Rotation Error:** red dashed at `5°` — common acceptable threshold
- **ADD Success:** red dashed at `90%` — target success rate

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Pose Estimation Evaluation - Test Set', fontsize=16, fontweight='bold')

classes = df_test['Class'].tolist()
x       = range(len(classes))

ax = axes[0, 0]
ax.bar(x, df_test['Trans RMSE (m)'], color='steelblue', edgecolor='k', linewidth=0.5)
ax.axhline(0.02, color='red', linestyle='--', linewidth=1.2, label='2 cm threshold')
ax.set_xticks(x); ax.set_xticklabels(classes, fontsize=9)
ax.set_ylabel('RMSE (m)'); ax.set_title('Translation RMSE')
ax.legend(fontsize=8)

ax = axes[0, 1]
ax.bar(x, df_test['Rot Error (deg)'], color='darkorange', edgecolor='k', linewidth=0.5)
ax.axhline(5.0, color='red', linestyle='--', linewidth=1.2, label='5 deg threshold')
ax.set_xticks(x); ax.set_xticklabels(classes, fontsize=9)
ax.set_ylabel('Degrees'); ax.set_title('Rotation Error')
ax.legend(fontsize=8)

ax = axes[1, 0]
ax.bar(x, df_test['Points MSE'], color='mediumseagreen', edgecolor='k', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(classes, fontsize=9)
ax.set_ylabel('MSE (m^2)'); ax.set_title('Radius Map Points MSE')

ax = axes[1, 1]
add_vals = df_test['ADD Success %'].fillna(0).tolist()
ax.bar(x, add_vals, color='mediumpurple', edgecolor='k', linewidth=0.5)
ax.axhline(90, color='red', linestyle='--', linewidth=1.2, label='90% target')
ax.set_ylim(0, 105)
ax.set_xticks(x); ax.set_xticklabels(classes, fontsize=9)
ax.set_ylabel('%'); ax.set_title('ADD Success Rate')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/content/evaluation_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to /content/evaluation_results.png')

---
## ✅ Evaluation Complete

- ✅ Validation metrics computed per class and overall
- ✅ Test metrics computed (final numbers)
- ✅ Metric bar charts saved to `/content/evaluation_results.png`

**Next step → Open and run `06_visualize.ipynb`**